#### Code

In [1]:
from pathlib import Path
import json

def data_prep(path_pd, path_kt, filetype="csv", a_exists=True, th=0.0001):
    fpath_pd = Path(path_pd)
    fpath_kt = Path(path_kt)

    data_pd, label_pd, data_kt, label_kt = [],[],[],[]
        
    if filetype=="csv":
        data_pd, label_pd = load_csv_with_labels(fpath_pd, "PD", a_exists=a_exists)
        data_kt, label_kt = load_csv_with_labels(fpath_kt, "KT", a_exists=a_exists)

    else:
        data_pd, label_pd = load_json_with_labels(fpath_pd, "PD", a_exists=a_exists)
        data_kt, label_kt = load_json_with_labels(fpath_kt, "KT", a_exists=a_exists)

    data = [*data_pd, *data_kt]
    labels = [*label_pd, *label_kt]

    data = cleaner(data, th)
    
    return data, labels

def cleaner(data, th):
    clean_data = []
    for sample in data:
        pts = []
        prev_t = 0
        for section in sample:
            for point in section:
                if point['t'] - prev_t > th:
                    pts.append(point)
                prev_t = point['t']
        clean_data.append(pts)
    return clean_data

def load_csv_with_labels(file_path, label, a_exists=True):
    import pandas as pd
    data = []
    labels = []
    for file in file_path.glob("*.csv"):
        df = pd.read_csv(file)
        if a_exists:
            df.columns = ['a', 'x', 'y', 'p', 't']
            df = df.drop(columns=['a'])
        else:
            df.columns = ['x', 'y', 'p', 't']
        points = df.to_dict(orient="records")
        data.append([points])
        labels.append(label)
    return data, labels

def load_json_with_labels(file_path, label, a_exists=True):
    data = []
    labels = []
    for file in file_path.glob("*.json"):
        with open(file, "r") as f:
            json_data = json.load(f)
        samples = json_data['data']
        if a_exists:
            samples = [
                [{k: v for k, v in point.items() if k != 'a'} for point in section]
                for section in samples
            ]
        data.append(samples)
        labels.append(label)
    return data, labels

# Raw data to arrays

In [2]:
data, labels = data_prep("./PD/spiral","./KT/spiral", "json")

__________
##### Code

In [3]:
import numpy as np
from scipy.fft import fft, fftfreq


def data_to_tab(data, sampling_rate=None, spiral_template=None):
    """
    Convert raw drawing samples to a tabular feature matrix.

    Parameters
    ----------
    data : list of lists of dicts
        Each dict must have keys: 'x', 'y', 'p', 't'. Key 'a' is optional.
    sampling_rate : float or None
        If None, estimated per-sample from median timestamp delta.
    spiral_template : list of (x, y) tuples or None
        If provided, computes spatial deviation from the template.

    Returns
    -------
    tab_data : np.ndarray of shape (n_samples, n_features)
    feature_labels : list of str, length n_features
    """

    tab_data = []
    feature_labels = None

    for sample in data:
        x = np.array([p['x'] for p in sample], dtype=float)
        y = np.array([p['y'] for p in sample], dtype=float)
        pr = np.array([p['p'] for p in sample], dtype=float)
        t = np.array([p['t'] for p in sample], dtype=float)
        has_a = 'a' in sample[0]
        if has_a:
            a = np.array([p['a'] for p in sample], dtype=float)

        dt = np.diff(t)
        dt = np.where(dt <= 0, 1e-9, dt)
        fs = sampling_rate if sampling_rate is not None else 1.0 / float(np.median(dt))

        dx = np.diff(x)
        dy = np.diff(y)
        seg_dists = np.sqrt(dx**2 + dy**2)
        velocity = seg_dists / dt
        alphas = np.arctan2(dy, dx)
        accel = np.diff(velocity) / dt[:-1]
        jerk = np.diff(accel) / dt[:-2]
        angular_velocity = np.diff(alphas) / dt[:-1]
        angular_accel = np.diff(angular_velocity) / dt[:-2]

        feats, labs = [], []

        def add_stats(arr, name):
            arr = np.array(arr, dtype=float)
            arr = arr[np.isfinite(arr)]
            if len(arr) == 0:
                arr = np.array([0.0])
            feats.extend([arr.max(), arr.min(), arr.mean(), np.median(arr)])
            labs.extend([f"{name}_max", f"{name}_min", f"{name}_mean", f"{name}_median"])

        def add_mass(arr, name):
            arr = np.array(arr, dtype=float)
            feats.append(float(np.sum(np.abs(arr[np.isfinite(arr)]))))
            labs.append(f"{name}_mass")

        def add_val(val, name):
            feats.append(float(val))
            labs.append(name)

        # --- global ---
        total_dist = float(np.sum(seg_dists))
        endpoint_dist = float(np.sqrt((x[-1]-x[0])**2 + (y[-1]-y[0])**2))
        add_val(t[-1] - t[0], 'duration')
        add_val(total_dist, 'total_distance')
        add_val(endpoint_dist / total_dist if total_dist > 0 else 0.0, 'straightness')

        # --- velocity ---
        add_stats(velocity, 'velocity')
        add_val(np.std(velocity) / np.mean(velocity) if np.mean(velocity) > 0 else 0.0, 'velocity_cv')

        # --- acceleration & jerk ---
        add_stats(accel, 'accel')
        add_mass(accel, 'accel')
        add_stats(jerk, 'jerk')
        add_mass(jerk, 'jerk')

        # --- angles ---
        add_stats(alphas, 'alpha')
        add_stats(angular_velocity, 'angular_velocity')
        add_stats(angular_accel, 'angular_accel')

        # --- pressure ---
        pressure_vel = np.diff(pr) / dt
        pressure_accel = np.diff(pressure_vel) / dt[:-1]
        add_stats(pr, 'pressure')
        add_stats(pressure_vel, 'pressure_velocity')
        add_stats(pressure_accel, 'pressure_accel')
        add_val(np.std(pr) / np.mean(pr) if np.mean(pr) > 0 else 0.0, 'pressure_cv')

        # --- altitude (only if 'a' present) ---
        if has_a:
            add_stats(a, 'altitude')
            add_stats(np.diff(a) / dt, 'altitude_velocity')

        # --- tremor (FFT of velocity) ---
        t_uniform = np.linspace(t[0], t[-1], len(t))
        vel_interp = np.interp(t_uniform, t[:-1] + dt/2, velocity)
        freqs = fftfreq(len(vel_interp), d=1.0/fs)
        fft_mag = np.abs(fft(vel_interp))[:len(vel_interp)//2]
        freqs_pos = freqs[:len(vel_interp)//2]

        tremor_mask = (freqs_pos >= 3.0) & (freqs_pos <= 12.0)
        if tremor_mask.any():
            tremor_power = float(np.sum(fft_mag[tremor_mask]**2))
            dominant_freq = float(freqs_pos[tremor_mask][np.argmax(fft_mag[tremor_mask])])
        else:
            tremor_power, dominant_freq = 0.0, 0.0

        total_power = float(np.sum(fft_mag**2))
        add_val(tremor_power, 'tremor_power')
        add_val(dominant_freq, 'dominant_tremor_freq')
        add_val(tremor_power / total_power if total_power > 0 else 0.0, 'tremor_ratio')

        # --- template deviation ---
        if spiral_template is not None:
            tx = np.array([pt[0] for pt in spiral_template], dtype=float)
            ty = np.array([pt[1] for pt in spiral_template], dtype=float)
            deviations = [float(np.min(np.sqrt((tx-xi)**2 + (ty-yi)**2))) for xi, yi in zip(x, y)]
            add_stats(deviations, 'template_deviation')

        if feature_labels is None:
            feature_labels = labs
        elif labs != feature_labels:
            raise ValueError(f"Feature mismatch: expected {len(feature_labels)}, got {len(labs)} features. "
                             "Ensure 'a' presence is consistent across all samples.")

        tab_data.append(feats)

    return np.array(tab_data, dtype=float), feature_labels

# Data to Tabular Features

In [4]:
features, feature_labels = data_to_tab(data)

_____________
##### Code

In [5]:
import joblib
import numpy as np
from sklearn.feature_selection import VarianceThreshold, SelectKBest


def _fisher_score(X, y):
    X, y = np.asarray(X, dtype=float), np.asarray(y)
    classes, y_idx = np.unique(y, return_inverse=True)
    mu_global = X.mean(axis=0)
    num = np.zeros(X.shape[1])
    den = np.zeros(X.shape[1])
    for k in range(len(classes)):
        Xk = X[y_idx == k]
        nk = len(Xk)
        if nk == 0:
            continue
        diff = Xk.mean(axis=0) - mu_global
        num += nk * diff**2
        den += nk * Xk.var(axis=0, ddof=0)
    scores = np.where(den > 0, num / den, 0.0)
    return scores, np.zeros_like(scores)


def feature_ext_tab(
    tab_data,
    labels,
    feature_names,
    eval=False,
    k_select=None,
    variance_threshold=0.01,
    selector_path="tab_feature_selectors.pkl",
):
    """
    Two-stage feature selection: VarianceThreshold then Fisher score.

    Parameters
    ----------
    tab_data : array-like (n_samples, n_features)
    labels : array-like (n_samples,)
    feature_names : list of str — pass directly from data_to_tab()
    eval : bool — if True, load and apply saved selectors (inference mode)
    k_select : int or None — features to keep; None uses elbow heuristic
    variance_threshold : float
    selector_path : str — where to save/load fitted selectors

    Returns
    -------
    X_selected : np.ndarray (n_samples, k)
    selected_names : list of str
    fisher_scores : list of (name, score) tuples, ranked descending
    """
    X = np.asarray(tab_data, dtype=float)
    y = np.asarray(labels)

    if len(feature_names) != X.shape[1]:
        raise ValueError(f"feature_names length {len(feature_names)} != tab_data columns {X.shape[1]}")

    # impute NaN/inf with column means
    X = np.where(np.isinf(X), np.nan, X)
    col_means = np.nanmean(X, axis=0)
    nan_locs = np.isnan(X)
    if nan_locs.any():
        X[nan_locs] = np.take(col_means, np.where(nan_locs)[1])

    # --- eval mode ---
    if eval:
        saved = joblib.load(selector_path)
        if list(feature_names) != saved["feature_names_original"]:
            raise ValueError("feature_names don't match those used at training time. Refit selectors.")

        X_post_var = saved["var_sel"].transform(X)
        X_selected = saved["kbest"].transform(X_post_var)
        mask = saved["kbest"].get_support()
        selected_names = [n for n, m in zip(saved["feature_names_post_var"], mask) if m]
        fisher_scores = sorted(
            zip(saved["feature_names_post_var"], saved["kbest"].scores_),
            key=lambda x: x[1], reverse=True
        )
        return X_selected, selected_names, fisher_scores

    # --- training mode ---

    # stage 1: variance threshold
    var_sel = VarianceThreshold(threshold=variance_threshold)
    X_post_var = var_sel.fit_transform(X)
    mask1 = var_sel.get_support()
    names_post_var = [n for n, m in zip(feature_names, mask1) if m]

    # stage 2: fisher scores on all surviving features
    kbest_full = SelectKBest(_fisher_score, k=X_post_var.shape[1])
    kbest_full.fit(X_post_var, y)
    fisher_scores = sorted(
        zip(names_post_var, kbest_full.scores_),
        key=lambda x: x[1], reverse=True
    )

    # elbow heuristic for k
    if k_select is None:
        ranked_scores = np.array([s for _, s in fisher_scores])
        cumulative = np.cumsum(ranked_scores) / np.sum(ranked_scores)
        k_select = int(np.searchsorted(cumulative, 0.80)) + 1
        k_select = np.clip(k_select, 5, min(15, X_post_var.shape[1]))

    # refit with chosen k
    kbest = SelectKBest(_fisher_score, k=k_select)
    kbest.fit(X_post_var, y)
    X_selected = kbest.transform(X_post_var)
    selected_names = [n for n, m in zip(names_post_var, kbest.get_support()) if m]

    joblib.dump({
        "var_sel":                var_sel,
        "kbest":                  kbest,
        "feature_names_original": list(feature_names),
        "feature_names_post_var": names_post_var,
        "k_select":               k_select,
        "fisher_scores_ranked":   fisher_scores,
    }, selector_path)

    return X_selected, selected_names, fisher_scores

# Tabular Feature Extraction

In [6]:
from sklearn.utils import shuffle
data_shuffled, tab_features_shuffled, labels_shuffled = shuffle(data, features, labels, random_state=42)
tab_features, selected_names, all_fisher_scores = feature_ext_tab(tab_features_shuffled, labels_shuffled, feature_labels, k_select=4)

________________
#### Code

In [7]:
import os
import numpy as np
import cv2


def _remove_close_points(sample, threshold):
    if len(sample) == 0:
        return np.empty((0, 2))

    coords = np.array([[p['x'], p['y']] for p in sample], dtype=float) \
             if isinstance(sample[0], dict) \
             else np.asarray(sample, dtype=float)[:, :2]

    if len(coords) < 2 or threshold <= 0:
        return coords

    kept = [coords[0]]
    for pt in coords[1:]:
        if np.linalg.norm(pt - kept[-1]) >= threshold:
            kept.append(pt)
    return np.array(kept)


def _normalise_coords(pts, canvas_size, margin=10):
    x, y = pts[:, 0], pts[:, 1]
    x_range = max(x.max() - x.min(), 1.0)
    y_range = max(y.max() - y.min(), 1.0)
    drawable = canvas_size - 2 * margin
    scale = drawable / max(x_range, y_range)

    x_px = ((x - x.min()) * scale).astype(np.int32)
    y_px = ((y - y.min()) * scale).astype(np.int32)
    x_px += margin + (drawable - x_px.max()) // 2
    y_px += margin + (drawable - y_px.max()) // 2
    return x_px, y_px


def data_to_img(data, labels, folder, canvas_size=256,
                close_point_threshold=1.0, max_pixel_gap=50, thickness=2):
    """
    Convert raw drawing samples to grayscale spiral images.

    Parameters
    ----------
    data : list of samples — dicts with 'x','y','p','t' or arrays (n, >=2)
    labels : list — class label per sample, used in output filename
    folder : str — output directory, created if needed
    canvas_size : int — output image size in pixels (square)
    close_point_threshold : float — min distance between consecutive points; 0 disables
    max_pixel_gap : int — segments longer than this are treated as pen-lifts
    thickness : int — line thickness in pixels

    Returns
    -------
    fnames : list of str — paths to saved PNG files
    failed : list of int — indices of samples with fewer than 2 points after filtering
    """
    os.makedirs(folder, exist_ok=True)
    fnames, failed = [], []

    for i, sample in enumerate(data):
        pts = _remove_close_points(sample, close_point_threshold)
        img = np.full((canvas_size, canvas_size), 255, dtype=np.uint8)

        if len(pts) >= 2:
            x_px, y_px = _normalise_coords(pts, canvas_size)
            for j in range(len(x_px) - 1):
                pt1 = (int(x_px[j]),     int(y_px[j]))
                pt2 = (int(x_px[j + 1]), int(y_px[j + 1]))
                if np.linalg.norm(np.array(pt2) - np.array(pt1)) <= max_pixel_gap:
                    cv2.line(img, pt1, pt2, color=0, thickness=thickness)
        else:
            failed.append(i)

        fname = os.path.join(folder, f"spiral_{i}_{labels[i]}.png")
        cv2.imwrite(fname, img)
        fnames.append(fname)

    return fnames, failed


# Data to Img

In [8]:
img_data_filenames, failed = data_to_img(data_shuffled, labels_shuffled, "IMG_0505_drawritepd")

____________
#### Code

In [14]:
import joblib
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import csv


def ext_feat_img(fnames, eval=False,
                 extractor_path="resnet18_feature_extractor.pt",
                 transform_path="image_transform.pkl"):
    """
    Extract 512-dim feature vectors from spiral images using a frozen ResNet18.

    Parameters
    ----------
    fnames : list of str — paths to PNG images
    eval : bool — if True, load saved transform; if False, fit and save

    Returns
    -------
    np.ndarray of shape (n_samples, 512)
    """
    img_transform = joblib.load(transform_path) if eval else transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])

    batch = torch.stack([img_transform(Image.open(f).convert("RGB")) for f in fnames])

    cnn = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    feature_extractor = nn.Sequential(*list(cnn.children())[:-1])
    feature_extractor.eval()
    for param in feature_extractor.parameters():
        param.requires_grad = False

    with torch.no_grad():
        feats = feature_extractor(batch).flatten(start_dim=1)

    if not eval:
        torch.save(feature_extractor.state_dict(), extractor_path)
        joblib.dump(img_transform, transform_path)

    return feats.numpy()


def img_downsize(X_img, eval=False, n_components=4,
                 selector_path="img_feature_selectors.pkl"):
    """
    Scale and reduce image features via PCA.

    Parameters
    ----------
    X_img : np.ndarray (n_samples, n_features)
    eval : bool — if True, load saved scaler/PCA; if False, fit and save
    n_components : int — number of PCA components

    Returns
    -------
    np.ndarray of shape (n_samples, n_components)
    """
    if eval:
        saved = joblib.load(selector_path)
        return saved["pca"].transform(saved["scaler"].transform(X_img))

    scaler = StandardScaler()
    pca = PCA(n_components=n_components)
    X_out = pca.fit_transform(scaler.fit_transform(X_img))
    joblib.dump({"scaler": scaler, "pca": pca}, selector_path)
    return X_out

def save_features_with_labels(features, labels, filename="all_features_050526.csv", header_prefix="f"):
    
    features = np.array(features)
    labels = np.array(labels).reshape(-1, 1)

    n_samples, n_features = features.shape
    headers = [f"{header_prefix}{i}" for i in range(n_features)] + ["label"]

    data = np.hstack((features, labels))

    with open(filename, mode="w", newline="") as file:
        writer = csv.writer(file)
        writer.writerow(headers)
        writer.writerows(data)

    print(f"Saved dataset with {n_samples} samples to {filename}")


# Ext Img Features

In [10]:
img_features = ext_feat_img(img_data_filenames)
img_features_downsized = img_downsize(img_features)

/Users/eleriin/msc/thesis/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:350: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/eleriin/msc/thesis/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:350: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/eleriin/msc/thesis/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:350: RuntimeWarning: invalid value encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/eleriin/msc/thesis/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:351: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users/eleriin/msc/thesis/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:351: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users/eleriin/msc/thesis/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:351: RuntimeWarning: invalid value encountered in matmul
  Q, _ = no

In [15]:
save_features_with_labels(img_features_downsized, labels_shuffled, "0505img_feat_54.csv")
save_features_with_labels(tab_features, labels_shuffled, "0505tab_feat_54.csv")

Saved dataset with 54 samples to 0505img_feat_54.csv
Saved dataset with 54 samples to 0505tab_feat_54.csv


_____________
#### Code

In [16]:
import numpy as np
import torch

def features_merge(X_tab, X_img):
    print("\nMerging tabular and image features...")
    #print("X_tab:", X_tab.shape)
    #print("X_img:", X_img.shape)

    X_merged = np.concatenate([X_tab, X_img], axis=1)

    print("Merged feature shape:", X_merged.shape)
    return torch.from_numpy(X_merged).float()
    


# Merge Features

In [17]:
features = features_merge(tab_features, img_features_downsized)


Merging tabular and image features...
Merged feature shape: (54, 8)


________
#### Code

In [26]:
import os
import json
import pickle
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from collections import Counter
from functools import partial
from pathlib import Path

from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, mutual_info_classif, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, ConfusionMatrixDisplay, make_scorer,
)
from sklearn.model_selection import (
    StratifiedKFold, GridSearchCV, cross_validate, train_test_split,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.primitives import Estimator
from qiskit.quantum_info import SparsePauliOp
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.algorithms.classifiers import NeuralNetworkClassifier
from qiskit_algorithms.optimizers import SPSA, COBYLA

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False


# ================================================================== #
# Shared utilities
# ================================================================== #

def _scale_to_pi(X_train, X_test, epsilon=1e-8):
    """Min-max scale train/test to [0, π] using train statistics."""
    X_min = X_train.min(axis=0)
    X_max = X_train.max(axis=0)
    scale = lambda X: (X - X_min) / (X_max - X_min + epsilon) * np.pi
    return scale(X_train), scale(X_test)


def _select_features(X_train, y_train, X_test, k):
    """Fit SelectKBest on train, transform both splits."""
    selector = SelectKBest(f_classif, k=k)
    X_train = selector.fit_transform(X_train, y_train)
    X_test  = selector.transform(X_test)
    return X_train, X_test


def _print_metrics(accs, precs, recs, f1s):
    print(f"Accuracy : {np.mean(accs):.3f} ± {np.std(accs):.3f}")
    print(f"Precision: {np.mean(precs):.3f} ± {np.std(precs):.3f}")
    print(f"Recall   : {np.mean(recs):.3f} ± {np.std(recs):.3f}")
    print(f"F1       : {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")


def _collect_metrics(y_test, y_pred):
    return (
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred, pos_label=-1, zero_division=0),
        recall_score(y_test,    y_pred, pos_label=-1, zero_division=0),
        f1_score(y_test,        y_pred, pos_label=-1, zero_division=0),
    )


def _make_observable(n_qubits, final_a=0, final_b=1):
    """Two-qubit observable: 0.5*Z_a + 0.5*Z_b + 0.5*Z_a Z_b."""
    terms = []
    for qa, qb in [(final_a, None), (final_b, None), (final_a, final_b)]:
        p = ["I"] * n_qubits
        p[qa] = "Z"
        if qb is not None:
            p[qb] = "Z"
        terms.append(("".join(p), 0.5))
    return SparsePauliOp.from_list(terms)


def _build_qnn(qc, x_params, weights, observable):
    return EstimatorQNN(
        circuit=qc,
        observables=observable,
        input_params=x_params,
        weight_params=weights,
        estimator=Estimator(),
    )


# ================================================================== #
# QCNN circuit
# ================================================================== #

def conv_pool_block(qc, a, b, conv_p, pool_p):
    """QCNN conv+pool on qubit pair (a, b). conv_p: 6 params, pool_p: 2 params."""
    qc.ry(conv_p[0], a); qc.rz(conv_p[1], a)
    qc.ry(conv_p[2], b); qc.rz(conv_p[3], b)
    qc.cx(a, b)
    qc.ry(conv_p[4], a); qc.rz(conv_p[5], b)
    qc.cx(b, a)
    qc.ry(pool_p[0], a); qc.rz(pool_p[1], a)
    qc.cx(a, b)


def build_qcnn_circuit(n_qubits, x_params, weights):
    """Hierarchical QCNN: compress to 2 qubits then apply readout block."""
    if 2 ** int(np.log2(n_qubits)) != n_qubits:
        raise ValueError(f"n_qubits must be a power of 2, got {n_qubits}.")

    qc = QuantumCircuit(n_qubits)
    for i in range(n_qubits):
        qc.ry(x_params[i], i)

    active, w = list(range(n_qubits)), 0
    while len(active) > 2:
        conv_p = weights[w:w+6]
        pool_p = weights[w+6:w+8]
        w += 8
        next_active = []
        for j in range(0, len(active), 2):
            conv_pool_block(qc, active[j], active[j+1], conv_p, pool_p)
            next_active.append(active[j])
        active = next_active

    a, b = active[0], active[1]
    read_p = weights[w:w+4]
    qc.ry(read_p[0], a); qc.rz(read_p[1], a)
    qc.ry(read_p[2], b); qc.rz(read_p[3], b)
    qc.cx(a, b)
    return qc


# ================================================================== #
# VQC circuit
# ================================================================== #

def build_vqc_circuit(n_qubits, x_params, weights, n_layers=2):
    """Angle encoding + n_layers of Ry rotations and ring CNOT entanglement."""
    qc = QuantumCircuit(n_qubits)
    for i in range(n_qubits):
        qc.ry(x_params[i], i)
    w = 0
    for _ in range(n_layers):
        for i in range(n_qubits):
            qc.ry(weights[w], i)
            w += 1
        for i in range(n_qubits):
            qc.cx(i, (i + 1) % n_qubits)
    return qc


def build_vqc_diagram(n_qubits=4, n_layers=2):
    """VQC circuit with barriers for visualisation only."""
    x_params = ParameterVector("x", n_qubits)
    weights  = ParameterVector("θ", n_layers * n_qubits)
    qc = QuantumCircuit(n_qubits)
    qc.barrier(label="Encode")
    for i in range(n_qubits):
        qc.ry(x_params[i], i)
    w = 0
    for layer in range(n_layers):
        qc.barrier(label=f"Layer {layer + 1}")
        for i in range(n_qubits):
            qc.ry(weights[w], i); w += 1
        for i in range(n_qubits):
            qc.cx(i, (i + 1) % n_qubits)
    qc.barrier(label="Z₀ obs")
    return qc


# ================================================================== #
# QCNN training callback
# ================================================================== #

loss_history_qcnn = []
params_history    = []

def _extract_scalar(args, kwargs):
    """Pull a numeric loss value from callback args/kwargs."""
    is_numeric = lambda v: (
        isinstance(v, (float, int, np.floating, np.integer))
        and not isinstance(v, (bool, np.bool_))
    )
    for key in ("value", "val", "loss", "fun", "f"):
        if key in kwargs and is_numeric(kwargs[key]):
            return float(kwargs[key])
    if len(args) >= 3 and is_numeric(args[2]):
        return float(args[2])
    for a in reversed(args):
        if is_numeric(a):
            return float(a)
    return None

def qcnn_training_callback(*args, **kwargs):
    global loss_history_qcnn, params_history
    if len(args) >= 2 and hasattr(args[1], "shape"):
        params_history.append(np.array(args[1], dtype=float))
    loss_history_qcnn.append(_extract_scalar(args, kwargs))


# ================================================================== #
# Visualisation
# ================================================================== #

def visualize_qcnn_results(X_test, y_test, y_pred, losses):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].plot(losses)
    axes[0].set(title="Training Loss", xlabel="Iteration", ylabel="Loss")

    X_proj = PCA(n_components=2).fit_transform(X_test)
    axes[1].scatter(X_proj[:, 0], X_proj[:, 1], c=y_pred, cmap="coolwarm", alpha=0.7)
    axes[1].set(title="Prediction Separation (PCA)", xlabel="PC1", ylabel="PC2")

    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[2],
                xticklabels=["HC", "PD"], yticklabels=["HC", "PD"])
    axes[2].set(title="Confusion Matrix", xlabel="Predicted", ylabel="True")

    plt.tight_layout()
    plt.show()


# ================================================================== #
# QCNN — single train/test run
# ================================================================== #

def exec_qcnn(X, y, epochs=150, model_path="qcnn_model.pkl"):
    """Train QCNN on an 80/20 split, save model, return path."""
    global loss_history_qcnn
    loss_history_qcnn = []

    if hasattr(X, "detach"):   # torch.Tensor
        X = X.detach().cpu().numpy()
    X = np.array(X)
    y = np.array([-1 if lab == "PD" else 1 for lab in y])

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    n_qubits = X.shape[1]
    n_stages = int(np.log2(n_qubits)) - 1
    x_params = ParameterVector("x", n_qubits)
    weights  = ParameterVector("θ", n_stages * 8 + 4)

    X_train, X_test = _scale_to_pi(X_train, X_test)

    qc         = build_qcnn_circuit(n_qubits, x_params, weights)
    observable = _make_observable(n_qubits)
    qnn        = _build_qnn(qc, x_params, weights, observable)

    classifier = NeuralNetworkClassifier(qnn, optimizer=SPSA(maxiter=epochs, callback=qcnn_training_callback))
    classifier.fit(X_train, y_train)

    losses_arr = np.array([v for v in loss_history_qcnn if v is not None], dtype=float)
    if losses_arr.size > 0:
        window = min(25, max(1, len(losses_arr) // 20))
        fig, ax = plt.subplots()
        ax.plot(losses_arr, label="loss")
        if window > 1:
            smooth = np.convolve(losses_arr, np.ones(window)/window, mode="valid")
            ax.plot(range(window-1, window-1+len(smooth)), smooth,
                    label=f"smooth(w={window})", linewidth=2)
        ax.set(title="QCNN Training Loss", xlabel="Iteration", ylabel="Loss")
        ax.legend(); ax.grid(True)
        plt.show()

    y_pred = classifier.predict(X_test)
    print(f"QCNN Accuracy: {accuracy_score(y_test, y_pred):.3f}")
    visualize_qcnn_results(X_test, y_test, y_pred, loss_history_qcnn)

    with open(model_path, "wb") as f:
        pickle.dump({"classifier": classifier,
                     "loss_history": loss_history_qcnn}, f)
    return model_path


# ================================================================== #
# QCNN — cross-validation
# ================================================================== #

def exec_qcnn_cv(X, y, epochs=150, n_splits=5, k_features=4):
    """Stratified k-fold CV for QCNN."""
    X = np.array(X)
    y = np.array([-1 if lab == "PD" else 1 for lab in y])
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    accs, precs, recs, f1s = [], [], [], []

    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
        print(f"\n===== Fold {fold} =====")
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        X_train, X_test = _select_features(X_train, y_train, X_test, k_features)
        X_train, X_test = _scale_to_pi(X_train, X_test)

        n_qubits = X_train.shape[1]
        n_stages = int(np.log2(n_qubits)) - 1
        x_params = ParameterVector("x", n_qubits)
        weights  = ParameterVector("θ", n_stages * 8 + 4)

        qc         = build_qcnn_circuit(n_qubits, x_params, weights)
        observable = _make_observable(n_qubits)
        qnn        = _build_qnn(qc, x_params, weights, observable)

        classifier = NeuralNetworkClassifier(qnn, optimizer=COBYLA(maxiter=epochs))
        classifier.fit(X_train, y_train)
        y_pred = classifier.predict(X_test)

        acc, prec, rec, f1 = _collect_metrics(y_test, y_pred)
        accs.append(acc); precs.append(prec); recs.append(rec); f1s.append(f1)
        print(f"Accuracy: {acc:.3f}  Precision: {prec:.3f}  Recall: {rec:.3f}  F1: {f1:.3f}")

    print("\n===== QCNN CV Results =====")
    _print_metrics(accs, precs, recs, f1s)
    return {"accuracy":  (np.mean(accs),  np.std(accs)),
            "precision": (np.mean(precs), np.std(precs)),
            "recall":    (np.mean(recs),  np.std(recs)),
            "f1":        (np.mean(f1s),   np.std(f1s))}


# ================================================================== #
# VQC — cross-validation
# ================================================================== #

def exec_vqc_cv(X, y, epochs=150, n_splits=5, n_layers=2, k_features=4):
    """Stratified k-fold CV for VQC."""
    X = np.array(X)
    y = np.array([-1 if lab == "PD" else 1 for lab in y])
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    accs, precs, recs, f1s = [], [], [], []

    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
        print(f"\n===== Fold {fold} =====")
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        X_train, X_test = _select_features(X_train, y_train, X_test, k_features)
        X_train, X_test = _scale_to_pi(X_train, X_test)

        n_qubits = X_train.shape[1]
        x_params = ParameterVector("x", n_qubits)
        weights  = ParameterVector("θ", n_layers * n_qubits)

        qc = build_vqc_circuit(n_qubits, x_params, weights, n_layers)

        pauli_str = ["I"] * n_qubits; pauli_str[0] = "Z"
        observable = SparsePauliOp.from_list([("".join(pauli_str), 1.0)])
        qnn = _build_qnn(qc, x_params, weights, observable)

        classifier = NeuralNetworkClassifier(qnn, optimizer=COBYLA(maxiter=epochs))
        classifier.fit(X_train, y_train)
        y_pred = classifier.predict(X_test)

        acc, prec, rec, f1 = _collect_metrics(y_test, y_pred)
        accs.append(acc); precs.append(prec); recs.append(rec); f1s.append(f1)
        print(f"Accuracy: {acc:.3f}  Precision: {prec:.3f}  Recall: {rec:.3f}  F1: {f1:.3f}")

    print("\n===== VQC CV Results =====")
    _print_metrics(accs, precs, recs, f1s)
    return {"accuracy":  (np.mean(accs),  np.std(accs)),
            "precision": (np.mean(precs), np.std(precs)),
            "recall":    (np.mean(recs),  np.std(recs)),
            "f1":        (np.mean(f1s),   np.std(f1s))}


# ================================================================== #
# Classical models — nested CV
# ================================================================== #

def _bootstrap_ci(y_true, y_pred, metric_fn, n_boot=2000, ci=95, seed=0):
    rng = np.random.default_rng(seed)
    n   = len(y_true)
    scores = [metric_fn(y_true[idx := rng.integers(0, n, n)],
                        y_pred[idx]) for _ in range(n_boot)]
    lo = (100 - ci) / 2
    return float(np.percentile(scores, lo)), float(np.percentile(scores, 100 - lo))


def _build_pipelines_and_grids(random_state, scale_pos_weight=1.0):
    def pipe(clf):
        return Pipeline([("scaler", StandardScaler()),
                         ("kbest",  SelectKBest(mutual_info_classif, k=6)),
                         ("clf",    clf)])

    pipelines = {
        "LR":  pipe(LogisticRegression(max_iter=2000, class_weight="balanced",
                                        solver="liblinear", random_state=random_state)),
        "SVM": pipe(SVC(probability=True, class_weight="balanced", random_state=random_state)),
        "RF":  pipe(RandomForestClassifier(class_weight="balanced", random_state=random_state)),
        "DT":  pipe(DecisionTreeClassifier(class_weight="balanced", random_state=random_state)),
        "KNN": pipe(KNeighborsClassifier()),
    }
    grids = {
        "LR":  {"kbest__k": [4, 6, 8, "all"], "clf__C": [0.01, 0.1, 1, 10],
                "clf__penalty": ["l1", "l2"]},
        "SVM": {"kbest__k": [4, 6, 8, "all"], "clf__kernel": ["rbf", "linear"],
                "clf__C": [0.1, 1, 10], "clf__gamma": ["scale", "auto"]},
        "RF":  {"kbest__k": [6, 8, "all"], "clf__n_estimators": [100, 200, 300],
                "clf__max_depth": [None, 5, 10], "clf__min_samples_leaf": [1, 2, 3]},
        "DT":  {"kbest__k": [4, 6, 8, "all"], "clf__max_depth": [None, 3, 5, 10],
                "clf__min_samples_leaf": [1, 2, 3, 5], "clf__criterion": ["gini", "entropy"]},
        "KNN": {"kbest__k": [4, 6, 8, "all"], "clf__n_neighbors": [3, 5, 7, 9],
                "clf__weights": ["uniform", "distance"], "clf__metric": ["euclidean", "manhattan"]},
    }
    if XGBOOST_AVAILABLE:
        pipelines["XGB"] = pipe(XGBClassifier(eval_metric="logloss", verbosity=0,
                                               random_state=random_state))
        grids["XGB"] = {"kbest__k": [6, 8, "all"], "clf__n_estimators": [100, 200],
                        "clf__max_depth": [3, 5], "clf__learning_rate": [0.05, 0.1, 0.2],
                        "clf__scale_pos_weight": [scale_pos_weight]}
    return pipelines, grids


def run_nested_cv(X, y_labels, feature_names=None, save_dir="models",
                  outer_folds=5, inner_folds=3, random_state=42, plot_confusion=True):
    """
    Nested stratified CV with GridSearchCV for LR, SVM, RF, DT, KNN (+ XGB if available).
    Outer loop estimates performance; inner loop tunes hyperparameters.
    Tuning criterion: F1 for PD class (pos_label=0).

    Returns dict of per-model metrics, CIs, confusion matrices,
    feature stability, best params, and saved model paths.
    """
    X = np.asarray(X, dtype=float)
    y = np.array([0 if lab == "PD" else 1 for lab in y_labels])
    os.makedirs(save_dir, exist_ok=True)

    n_pos, n_neg = int((y == 0).sum()), int((y == 1).sum())
    spw = n_neg / n_pos if n_pos > 0 else 1.0

    pipelines, grids = _build_pipelines_and_grids(random_state, scale_pos_weight=spw)
    outer_cv   = StratifiedKFold(n_splits=outer_folds, shuffle=True, random_state=random_state)
    inner_cv   = StratifiedKFold(n_splits=inner_folds, shuffle=True, random_state=random_state)
    fold_splits = list(outer_cv.split(X, y))

    scoring = {
        "accuracy":  make_scorer(accuracy_score),
        "precision": make_scorer(precision_score, pos_label=0, zero_division=0),
        "recall":    make_scorer(recall_score,    pos_label=0, zero_division=0),
        "f1":        make_scorer(f1_score,        pos_label=0, zero_division=0),
    }

    results         = {}
    all_model_preds = {}

    for name, pipe in pipelines.items():
        print(f"\n{'='*50}\n  {name}\n{'='*50}")

        grid = GridSearchCV(pipe, grids[name], scoring="f1", cv=inner_cv,
                            n_jobs=-1, refit=True, error_score="raise")
        outer_scores = cross_validate(grid, X, y, cv=outer_cv, scoring=scoring,
                                      return_estimator=True, n_jobs=1)

        sample_preds = np.full(len(y), -1, dtype=int)
        sample_probs = np.full(len(y), np.nan)
        best_params_per_fold = []
        features_per_fold    = []

        for fold_idx, (_, test_idx) in enumerate(fold_splits):
            fitted = outer_scores["estimator"][fold_idx]
            sample_preds[test_idx] = fitted.predict(X[test_idx])
            try:
                sample_probs[test_idx] = fitted.predict_proba(X[test_idx])[:, 0]
            except AttributeError:
                pass
            best_params_per_fold.append(getattr(fitted, "best_params_", None))
            if feature_names is not None:
                try:
                    mask = fitted.best_estimator_.named_steps["kbest"].get_support()
                    features_per_fold.append([n for n, m in zip(feature_names, mask) if m])
                except Exception:
                    features_per_fold.append([])

        all_model_preds[name] = sample_preds.copy()
        accs  = outer_scores["test_accuracy"]
        precs = outer_scores["test_precision"]
        recs  = outer_scores["test_recall"]
        f1s   = outer_scores["test_f1"]

        acc_ci = _bootstrap_ci(y, sample_preds, accuracy_score, seed=random_state)
        f1_ci  = _bootstrap_ci(y, sample_preds,
                               partial(f1_score, pos_label=0, zero_division=0),
                               seed=random_state)

        cm = confusion_matrix(y, sample_preds, labels=[0, 1])
        if plot_confusion:
            fig, ax = plt.subplots(figsize=(4, 4))
            ConfusionMatrixDisplay(cm, display_labels=["PD", "HC"]).plot(ax=ax, colorbar=False)
            ax.set_title(f"{name} — outer folds")
            fig.tight_layout()
            fig.savefig(os.path.join(save_dir, f"{name}_confusion_matrix.png"), dpi=150)
            plt.close(fig)

        feature_stability = {}
        if features_per_fold:
            counts = Counter(f for fold in features_per_fold for f in fold)
            feature_stability = {feat: f"{c}/{outer_folds}" for feat, c in counts.most_common()}

        param_strs      = [str(p) for p in best_params_per_fold if p is not None]
        most_common_str = Counter(param_strs).most_common(1)[0][0] if param_strs else "{}"
        most_common_params = best_params_per_fold[param_strs.index(most_common_str)] \
                             if param_strs else {}

        final_pipe = pipelines[name].set_params(**most_common_params)
        final_pipe.fit(X, y)
        model_path = os.path.join(save_dir, f"{name}_final.joblib")
        joblib.dump(final_pipe, model_path)

        results[name] = {
            "accuracy_mean":  float(accs.mean()),  "accuracy_std":  float(accs.std()),
            "accuracy_ci":    list(acc_ci),
            "precision_mean": float(precs.mean()), "precision_std": float(precs.std()),
            "recall_mean":    float(recs.mean()),  "recall_std":    float(recs.std()),
            "f1_mean":        float(f1s.mean()),   "f1_std":        float(f1s.std()),
            "f1_ci":          list(f1_ci),
            "confusion_matrix":     cm.tolist(),
            "sample_predictions":   sample_preds.tolist(),
            "sample_probs_pd":      sample_probs.tolist(),
            "best_params_per_fold": best_params_per_fold,
            "most_common_params":   most_common_params,
            "feature_stability":    feature_stability,
            "saved_model":          model_path,
        }

    # hard-case analysis
    hard_cases = sorted([
        (idx,
         "PD" if y[idx] == 0 else "HC",
         sum(1 for p in all_model_preds.values() if p[idx] != y[idx]),
         {n: ("✓" if all_model_preds[n][idx] == y[idx] else "✗") for n in pipelines})
        for idx in range(len(y))
    ], key=lambda r: r[2], reverse=True)

    results["hard_cases"] = [
        {"sample_idx": idx, "true_label": lab,
         "n_models_wrong": wrong, "per_model": per}
        for idx, lab, wrong, per in hard_cases if wrong > 0
    ]

    summary_path = os.path.join(save_dir, "nested_cv_results.json")
    with open(summary_path, "w") as fh:
        json.dump(results, fh, indent=2)

    print(f"\n{'='*50}")
    print(f"  Final Results Summary")
    print(f"{'='*50}")
    print(f"  {'Model':<8} {'Acc':>6} {'±':>5} {'F1':>6} {'±':>5} {'Prec':>6} {'Rec':>6}  {'Acc CI':>15}  {'F1 CI':>15}")
    print(f"  {'-'*75}")
    for name, r in results.items():
        if name == "hard_cases":
            continue
        print(
            f"  {name:<8} "
            f"{r['accuracy_mean']:>6.3f} {r['accuracy_std']:>5.3f}  "
            f"{r['f1_mean']:>6.3f} {r['f1_std']:>5.3f}  "
            f"{r['precision_mean']:>6.3f}  {r['recall_mean']:>6.3f}  "
            f"[{r['accuracy_ci'][0]:.3f}, {r['accuracy_ci'][1]:.3f}]  "
            f"[{r['f1_ci'][0]:.3f}, {r['f1_ci'][1]:.3f}]"
        )

    if results.get("hard_cases"):
        print(f"\n  Hard cases (misclassified by at least one model):")
        print(f"  {'Idx':>4}  {'True':>5}  {'#Wrong':>6}  " +
              "  ".join(f"{n:>5}" for n in pipelines))
        print(f"  {'-'*50}")
        for case in results["hard_cases"]:
            cols = "  ".join(f"{case['per_model'].get(n, '?'):>5}" for n in pipelines)
            print(f"  {case['sample_idx']:>4}  {case['true_label']:>5}  "
                  f"{case['n_models_wrong']:>6}  {cols}")

    return results


# ================================================================== #
# Misc utilities
# ================================================================== #

def load_features_csv(filename, has_labels=True):
    import csv
    with open(filename) as f:
        reader = csv.reader(f)
        next(reader)  # skip header
        data = np.array(list(reader))
    if has_labels:
        return data[:, :-1].astype(float), data[:, -1]
    return data.astype(float), None


def save_model(X, y, classifier, classifier_name, method, save_dir="models"):
    from sklearn.feature_selection import RFE
    from sklearn.feature_selection import f_classif

    selector = (SelectKBest(f_classif, k=min(4, X.shape[1])) if method == "filter"
                else RFE(SVC(kernel="linear"), n_features_to_select=min(4, X.shape[1])))

    pipeline = Pipeline([("scaler", StandardScaler()),
                         ("selector", selector),
                         ("clf", classifier)])
    pipeline.fit(X, y)

    Path(save_dir).mkdir(exist_ok=True)
    path = f"{save_dir}/{method}_{classifier_name}.pkl"
    joblib.dump(pipeline, path)
    return path

In [19]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch

def figures():
       # ── colour palette ─────────────────────────────────────────────────────────────
       PURPLE = "#352b60"
       PINK   = "#e4057f"
       GREY   = "#888888"
       BLUE   = "#2196F3"

       # ── data ──────────────────────────────────────────────────────────────────────
       models = ["LR", "SVM", "RF", "XGB", "DT", "KNN", "VQC", "QCNN"]

       f1_tabular    = [0.877, 0.877, 0.818, 0.816, 0.742, 0.859, 0.831, 0.766]
       f1_image      = [0.571, 0.114, 0.509, 0.571, 0.502, 0.600, 0.404, 0.261]
       f1_multimodal = [0.877, 0.826, 0.836, 0.794, 0.742, 0.877, 0.818, 0.833]

       std_tabular    = [0.080, 0.080, 0.151, 0.131, 0.126, 0.085, 0.049, 0.093]
       std_image      = [0.054, 0.229, 0.063, 0.054, 0.090, 0.082, 0.102, 0.137]
       std_multimodal = [0.080, 0.113, 0.153, 0.104, 0.126, 0.080, 0.110, 0.115]

       # ── Figure 1: F1 across modalities ────────────────────────────────────────────
       x     = np.arange(len(models))
       width = 0.25

       fig, ax = plt.subplots(figsize=(13, 6))

       ax.bar(x - width, f1_tabular,    width,
              yerr=std_tabular,    capsize=3, color=PURPLE,
              error_kw=dict(ecolor="black", elinewidth=0.8), label="Tabular")
       ax.bar(x,          f1_image,      width,
              yerr=std_image,      capsize=3, color=GREY,
              error_kw=dict(ecolor="black", elinewidth=0.8), label="Image-only")
       ax.bar(x + width,  f1_multimodal, width,
              yerr=std_multimodal, capsize=3, color=PINK,
              error_kw=dict(ecolor="black", elinewidth=0.8), label="Multimodal")

       ax.axvline(x=5.5, color="black", linestyle="--", linewidth=0.8, alpha=0.4)
       ax.text(5.6, 0.95, "Quantum", fontsize=8, color="black", alpha=0.5)
       ax.text(0.0, 0.95, "Classical", fontsize=8, color="black", alpha=0.5)

       ax.set_xticks(x)
       ax.set_xticklabels(models)
       ax.set_ylabel("F1-score")
       ax.set_ylim(0, 1.05)
       ax.set_title("F1-score Across Feature Representations")
       ax.legend(loc="upper left", bbox_to_anchor=(1, 1))
       ax.grid(axis="y", alpha=0.4)

       plt.tight_layout()
       plt.savefig("f1scores.png", dpi=150)
       plt.show()

       # ── Figure 2: LR vs VQC vs QCNN ───────────────────────────────────────────────
       labels = ["Tabular", "Image-only", "Multimodal"]

       lr_f1   = [0.877, 0.571, 0.877]
       vqc_f1  = [0.831, 0.404, 0.818]
       qcnn_f1 = [0.766, 0.261, 0.833]

       lr_std   = [0.080, 0.054, 0.080]
       vqc_std  = [0.049, 0.102, 0.110]
       qcnn_std = [0.093, 0.137, 0.115]

       x     = np.arange(len(labels))
       width = 0.25

       fig, ax = plt.subplots(figsize=(9, 5))

       ax.bar(x - width, lr_f1,   width,
              yerr=lr_std,   capsize=4, color=PURPLE, label="LR (best classical)",
              error_kw=dict(ecolor="black", elinewidth=0.8))
       ax.bar(x,          vqc_f1,  width,
              yerr=vqc_std,  capsize=4, color=GREY,   label="VQC",
              error_kw=dict(ecolor="black", elinewidth=0.8))
       ax.bar(x + width,  qcnn_f1, width,
              yerr=qcnn_std, capsize=4, color=PINK,   label="QCNN",
              error_kw=dict(ecolor="black", elinewidth=0.8))

       ax.set_xticks(x)
       ax.set_xticklabels(labels)
       ax.set_ylabel("F1-score")
       ax.set_ylim(0, 1.05)
       ax.set_title("Quantum Models vs Best Classical Baseline")
       ax.legend(loc="upper left", bbox_to_anchor=(1, 1))
       ax.grid(axis="y", alpha=0.4)

       plt.tight_layout()
       plt.savefig("qcnn_vs_cl.png", dpi=150)
       plt.show()

       # ── Figure 3: stability (F1 std) ──────────────────────────────────────────────
       x     = np.arange(len(models))
       width = 0.25

       fig, ax = plt.subplots(figsize=(13, 5))

       ax.bar(x - width, std_tabular,    width, color=PURPLE, label="Tabular")
       ax.bar(x,          std_image,      width, color=GREY,   label="Image-only")
       ax.bar(x + width,  std_multimodal, width, color=PINK,   label="Multimodal")

       ax.axvline(x=5.5, color="black", linestyle="--", linewidth=0.8, alpha=0.4)
       ax.text(5.6, 0.005, "Quantum", fontsize=8, color="black", alpha=0.5)
       ax.text(0.0, 0.005, "Classical", fontsize=8, color="black", alpha=0.5)

       ax.set_xticks(x)
       ax.set_xticklabels(models)
       ax.set_ylabel("F1 standard deviation across folds")
       ax.set_title("Model Stability Across Feature Representations")
       ax.set_ylim(0, 0.30)
       ax.legend(loc="upper left", bbox_to_anchor=(1, 1))
       ax.grid(axis="y", alpha=0.4)

       plt.tight_layout()
       plt.savefig("variance.png", dpi=150)
       plt.show()


# Train Model

In [20]:
results = exec_qcnn_cv(features, labels_shuffled, epochs=300)

/var/folders/j7/tz4lqnhs41s1tcc_z0t0_19h0000gn/T/ipykernel_64118/3336539506.py:298: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  X = np.array(X)
/var/folders/j7/tz4lqnhs41s1tcc_z0t0_19h0000gn/T/ipykernel_64118/3336539506.py:99: DeprecationWarning: The class ``qiskit.primitives.estimator.Estimator`` is deprecated as of qiskit 1.2. It will be removed no earlier than 3 months after the release date. All implementations of the `BaseEstimatorV1` interface have been deprecated in favor of their V2 counterparts. The V2 alternative for the `Estimator` class is `StatevectorEstimator`.
  estimator=Estimator(),
/var/folders/j7/tz4lqnhs41s1tcc_z0t0_19h0000gn/T/ipykernel_64118/3336539506.py:94: DeprecationWarning: V1 Primitives are deprecate


===== Fold 1 =====
Accuracy: 0.727  Precision: 0.600  Recall: 0.750  F1: 0.667

===== Fold 2 =====


/var/folders/j7/tz4lqnhs41s1tcc_z0t0_19h0000gn/T/ipykernel_64118/3336539506.py:99: DeprecationWarning: The class ``qiskit.primitives.estimator.Estimator`` is deprecated as of qiskit 1.2. It will be removed no earlier than 3 months after the release date. All implementations of the `BaseEstimatorV1` interface have been deprecated in favor of their V2 counterparts. The V2 alternative for the `Estimator` class is `StatevectorEstimator`.
  estimator=Estimator(),
/var/folders/j7/tz4lqnhs41s1tcc_z0t0_19h0000gn/T/ipykernel_64118/3336539506.py:94: DeprecationWarning: V1 Primitives are deprecated as of qiskit-machine-learning 0.8.0 and will be removed no sooner than 4 months after the release date. Use V2 primitives for continued compatibility and support.
  return EstimatorQNN(


Accuracy: 0.909  Precision: 1.000  Recall: 0.750  F1: 0.857

===== Fold 3 =====


/var/folders/j7/tz4lqnhs41s1tcc_z0t0_19h0000gn/T/ipykernel_64118/3336539506.py:99: DeprecationWarning: The class ``qiskit.primitives.estimator.Estimator`` is deprecated as of qiskit 1.2. It will be removed no earlier than 3 months after the release date. All implementations of the `BaseEstimatorV1` interface have been deprecated in favor of their V2 counterparts. The V2 alternative for the `Estimator` class is `StatevectorEstimator`.
  estimator=Estimator(),
/var/folders/j7/tz4lqnhs41s1tcc_z0t0_19h0000gn/T/ipykernel_64118/3336539506.py:94: DeprecationWarning: V1 Primitives are deprecated as of qiskit-machine-learning 0.8.0 and will be removed no sooner than 4 months after the release date. Use V2 primitives for continued compatibility and support.
  return EstimatorQNN(


Accuracy: 0.909  Precision: 1.000  Recall: 0.750  F1: 0.857

===== Fold 4 =====


/var/folders/j7/tz4lqnhs41s1tcc_z0t0_19h0000gn/T/ipykernel_64118/3336539506.py:99: DeprecationWarning: The class ``qiskit.primitives.estimator.Estimator`` is deprecated as of qiskit 1.2. It will be removed no earlier than 3 months after the release date. All implementations of the `BaseEstimatorV1` interface have been deprecated in favor of their V2 counterparts. The V2 alternative for the `Estimator` class is `StatevectorEstimator`.
  estimator=Estimator(),
/var/folders/j7/tz4lqnhs41s1tcc_z0t0_19h0000gn/T/ipykernel_64118/3336539506.py:94: DeprecationWarning: V1 Primitives are deprecated as of qiskit-machine-learning 0.8.0 and will be removed no sooner than 4 months after the release date. Use V2 primitives for continued compatibility and support.
  return EstimatorQNN(


Accuracy: 0.636  Precision: 0.500  Recall: 0.750  F1: 0.600

===== Fold 5 =====


/var/folders/j7/tz4lqnhs41s1tcc_z0t0_19h0000gn/T/ipykernel_64118/3336539506.py:99: DeprecationWarning: The class ``qiskit.primitives.estimator.Estimator`` is deprecated as of qiskit 1.2. It will be removed no earlier than 3 months after the release date. All implementations of the `BaseEstimatorV1` interface have been deprecated in favor of their V2 counterparts. The V2 alternative for the `Estimator` class is `StatevectorEstimator`.
  estimator=Estimator(),
/var/folders/j7/tz4lqnhs41s1tcc_z0t0_19h0000gn/T/ipykernel_64118/3336539506.py:94: DeprecationWarning: V1 Primitives are deprecated as of qiskit-machine-learning 0.8.0 and will be removed no sooner than 4 months after the release date. Use V2 primitives for continued compatibility and support.
  return EstimatorQNN(


Accuracy: 0.900  Precision: 0.800  Recall: 1.000  F1: 0.889

===== QCNN CV Results =====
Accuracy : 0.816 ± 0.114
Precision: 0.780 ± 0.204
Recall   : 0.800 ± 0.100
F1       : 0.774 ± 0.117


In [21]:
results = exec_vqc_cv(features, labels_shuffled, epochs=300)

/var/folders/j7/tz4lqnhs41s1tcc_z0t0_19h0000gn/T/ipykernel_64118/3336539506.py:342: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  X = np.array(X)
/var/folders/j7/tz4lqnhs41s1tcc_z0t0_19h0000gn/T/ipykernel_64118/3336539506.py:99: DeprecationWarning: The class ``qiskit.primitives.estimator.Estimator`` is deprecated as of qiskit 1.2. It will be removed no earlier than 3 months after the release date. All implementations of the `BaseEstimatorV1` interface have been deprecated in favor of their V2 counterparts. The V2 alternative for the `Estimator` class is `StatevectorEstimator`.
  estimator=Estimator(),
/var/folders/j7/tz4lqnhs41s1tcc_z0t0_19h0000gn/T/ipykernel_64118/3336539506.py:94: DeprecationWarning: V1 Primitives are deprecate


===== Fold 1 =====
Accuracy: 0.727  Precision: 0.600  Recall: 0.750  F1: 0.667

===== Fold 2 =====


/var/folders/j7/tz4lqnhs41s1tcc_z0t0_19h0000gn/T/ipykernel_64118/3336539506.py:99: DeprecationWarning: The class ``qiskit.primitives.estimator.Estimator`` is deprecated as of qiskit 1.2. It will be removed no earlier than 3 months after the release date. All implementations of the `BaseEstimatorV1` interface have been deprecated in favor of their V2 counterparts. The V2 alternative for the `Estimator` class is `StatevectorEstimator`.
  estimator=Estimator(),
/var/folders/j7/tz4lqnhs41s1tcc_z0t0_19h0000gn/T/ipykernel_64118/3336539506.py:94: DeprecationWarning: V1 Primitives are deprecated as of qiskit-machine-learning 0.8.0 and will be removed no sooner than 4 months after the release date. Use V2 primitives for continued compatibility and support.
  return EstimatorQNN(


Accuracy: 0.909  Precision: 1.000  Recall: 0.750  F1: 0.857

===== Fold 3 =====


/var/folders/j7/tz4lqnhs41s1tcc_z0t0_19h0000gn/T/ipykernel_64118/3336539506.py:99: DeprecationWarning: The class ``qiskit.primitives.estimator.Estimator`` is deprecated as of qiskit 1.2. It will be removed no earlier than 3 months after the release date. All implementations of the `BaseEstimatorV1` interface have been deprecated in favor of their V2 counterparts. The V2 alternative for the `Estimator` class is `StatevectorEstimator`.
  estimator=Estimator(),
/var/folders/j7/tz4lqnhs41s1tcc_z0t0_19h0000gn/T/ipykernel_64118/3336539506.py:94: DeprecationWarning: V1 Primitives are deprecated as of qiskit-machine-learning 0.8.0 and will be removed no sooner than 4 months after the release date. Use V2 primitives for continued compatibility and support.
  return EstimatorQNN(


Accuracy: 0.909  Precision: 1.000  Recall: 0.750  F1: 0.857

===== Fold 4 =====


/var/folders/j7/tz4lqnhs41s1tcc_z0t0_19h0000gn/T/ipykernel_64118/3336539506.py:99: DeprecationWarning: The class ``qiskit.primitives.estimator.Estimator`` is deprecated as of qiskit 1.2. It will be removed no earlier than 3 months after the release date. All implementations of the `BaseEstimatorV1` interface have been deprecated in favor of their V2 counterparts. The V2 alternative for the `Estimator` class is `StatevectorEstimator`.
  estimator=Estimator(),
/var/folders/j7/tz4lqnhs41s1tcc_z0t0_19h0000gn/T/ipykernel_64118/3336539506.py:94: DeprecationWarning: V1 Primitives are deprecated as of qiskit-machine-learning 0.8.0 and will be removed no sooner than 4 months after the release date. Use V2 primitives for continued compatibility and support.
  return EstimatorQNN(


Accuracy: 0.727  Precision: 0.600  Recall: 0.750  F1: 0.667

===== Fold 5 =====


/var/folders/j7/tz4lqnhs41s1tcc_z0t0_19h0000gn/T/ipykernel_64118/3336539506.py:99: DeprecationWarning: The class ``qiskit.primitives.estimator.Estimator`` is deprecated as of qiskit 1.2. It will be removed no earlier than 3 months after the release date. All implementations of the `BaseEstimatorV1` interface have been deprecated in favor of their V2 counterparts. The V2 alternative for the `Estimator` class is `StatevectorEstimator`.
  estimator=Estimator(),
/var/folders/j7/tz4lqnhs41s1tcc_z0t0_19h0000gn/T/ipykernel_64118/3336539506.py:94: DeprecationWarning: V1 Primitives are deprecated as of qiskit-machine-learning 0.8.0 and will be removed no sooner than 4 months after the release date. Use V2 primitives for continued compatibility and support.
  return EstimatorQNN(


Accuracy: 0.900  Precision: 0.800  Recall: 1.000  F1: 0.889

===== VQC CV Results =====
Accuracy : 0.835 ± 0.088
Precision: 0.800 ± 0.179
Recall   : 0.800 ± 0.100
F1       : 0.787 ± 0.099


In [27]:
results = run_nested_cv(features, labels_shuffled, feature_names=feature_labels , save_dir="models_from_54", outer_folds=5)


  LR

  SVM

  RF

  DT

  KNN

  XGB

  Final Results Summary
  Model       Acc     ±     F1     ±   Prec    Rec           Acc CI            F1 CI
  ---------------------------------------------------------------------------
  LR        0.795 0.110   0.758 0.096   0.764   0.800  [0.685, 0.889]  [0.585, 0.878]
  SVM       0.818 0.152   0.783 0.158   0.800   0.800  [0.704, 0.907]  [0.600, 0.889]
  RF        0.871 0.123   0.847 0.133   0.820   0.900  [0.778, 0.944]  [0.703, 0.936]
  DT        0.760 0.147   0.742 0.126   0.696   0.850  [0.648, 0.870]  [0.558, 0.852]
  KNN       0.871 0.043   0.837 0.054   0.803   0.900  [0.778, 0.963]  [0.700, 0.945]
  XGB       0.871 0.043   0.827 0.064   0.820   0.850  [0.778, 0.945]  [0.686, 0.941]

  Hard cases (misclassified by at least one model):
   Idx   True  #Wrong     LR    SVM     RF     DT    KNN    XGB
  --------------------------------------------------
    37     PD       6      ✗      ✗      ✗      ✗      ✗      ✗
    43     PD       6  

____________
##### Code

In [28]:
def evaluate_model_get_feat(data_path_pd, data_path_kt, img_folder):
    from sklearn.utils import shuffle

    data, labels = data_prep(data_path_pd, data_path_kt)
    data, labels = shuffle(data, labels, random_state=42)

    tab_data, feature_labels = data_to_tab(data)
    tab_features, _, _ = feature_ext_tab(tab_data, labels, feature_labels, eval=True)

    img_fnames, _ = data_to_img(data, labels, img_folder)
    img_features  = img_downsize(ext_feat_img(img_fnames, eval=True), eval=True)

    return features_merge(tab_features, img_features), labels


def evaluate_model_predictions(model_path, features, labels):
    import joblib
    import numpy as np
    import matplotlib.pyplot as plt
    from sklearn.metrics import (
        accuracy_score, confusion_matrix,
        classification_report, roc_auc_score, roc_curve
    )

    model_bundle = joblib.load(model_path)
    classifier   = model_bundle["classifier"]
    x_min        = model_bundle["x_min"]
    x_max        = model_bundle["x_max"]

    features = np.array(features)
    features_scaled = (features - x_min) / (x_max - x_min + 1e-8) * np.pi
    y_true = np.array([-1 if lab == "PD" else 1 for lab in labels])
    y_pred = classifier.predict(features_scaled)

    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()

    precision   = tp / (tp + fp) if (tp + fp) else 0
    recall      = tp / (tp + fn) if (tp + fn) else 0
    specificity = tn / (tn + fp) if (tn + fp) else 0
    f1          = 2 * precision * recall / (precision + recall) if (precision + recall) else 0

    print(f"Accuracy   : {accuracy_score(y_true, y_pred):.3f}")
    print(f"Precision  : {precision:.3f}")
    print(f"Recall     : {recall:.3f}")
    print(f"Specificity: {specificity:.3f}")
    print(f"F1         : {f1:.3f}")
    print(f"\nConfusion Matrix:\n{cm}")
    print(f"\nClassification Report:\n{classification_report(y_true, y_pred)}")

    # ROC — quantum models output discrete {-1, 1} so AUC will be a single point,
    # not a curve. Use predict_proba if available, otherwise keep as-is.
    try:
        y_scores = classifier.predict_proba(features_scaled)[:, 1]
    except AttributeError:
        y_scores = (y_pred + 1) / 2   # fallback: discrete {-1,1} → {0,1}

    y_binary = (y_true + 1) / 2
    auc = roc_auc_score(y_binary, y_scores)
    fpr, tpr, _ = roc_curve(y_binary, y_scores)

    print(f"ROC AUC    : {auc:.3f}")

    plt.figure()
    plt.plot(fpr, tpr, label=f"AUC = {auc:.3f}")
    plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve")
    plt.legend()
    plt.grid()
    plt.show()

_________
# Evaluate




In [ ]:
eval_features, eval_labels = evaluate_model_get_feat("./processed-files-sven/PD/","./processed-files-sven/KT/","IMG_880_eval")
evaluate_model_predictions("qcnn_model_v2_1604.pkl", eval_features, eval_labels)